
Forward pass - start with the smallest components that have the least (zero) dependencies and build up documentation as we move toward higher level modules.  The idea here is to look at how things work and deduce what they do.


In [1]:
import json
from pathlib import Path
from typing import List, Dict, Set, Tuple
import networkx as nx
import sys
import os
from pprint import pprint


os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)

# --- Config ---
PROJECT_ROOT = Path('../..').resolve()  # Adjust as needed to find the project root
GRAPH_PATH = PROJECT_ROOT / ".ai/context/internal/code_graph.gml"
DATABASE_PATH = PROJECT_ROOT / ".ai/context/internal/code_graph.json"
PRIORITY_PATH = PROJECT_ROOT / ".ai/context/internal/forward_pass_schedule.json"
DOX_DIR = PROJECT_ROOT / ".ai/context/internal/generated_dox"
XML_DIR = PROJECT_ROOT / ".ai/context/doxygen_output/xml"


# Add the modules directory to the Python path if needed
modules_dir = PROJECT_ROOT / '.ai/modules'
if str(modules_dir) not in sys.path:
    sys.path.append(str(modules_dir))

SRC_DIR = PROJECT_ROOT / 'src'
TEMP_DIR = PROJECT_ROOT / 'tmp'

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/qte2333/repos/legacy


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import doxygen_parse as dp

db = dp.load_db(DATABASE_PATH)

In [4]:
import doxygen_graph as dg

# --- Load graph and priority ---
graph: nx.DiGraph = dg.load_graph(GRAPH_PATH)

2025-06-22 07:15:30.825 | INFO     | doxygen_graph:load_graph:462 - Graph loaded from /Users/qte2333/repos/legacy/.ai/context/internal/code_graph.gml, nodes: 14517, edges: 50282


In [5]:
import doc_db

In [6]:
# for the second forward pass, we can summarize more than just functions and
# include entities like enums, classes, structs, etc that are referenced in
# implementation and we should now understand the purpose of.

# add nodes we don't want to document to the visited set
skip = set(
    n for n,d in graph.nodes(data=True)
#    if d.get("type") in (dg.EntityType.BODY, dg.EntityType.DECLARATION)
    if d.get('type') not in (
        dg.EntityType.COMPOUND, dg.EntityType.MEMBER
    ) or d.get('kind') not in (
        'variable',
        'enum',
        'function',
        'define',
        'struct',
        'class',
        'typedef',
        'group',
        'namespace',
    )
)

priority_order = dg.create_visit_list(graph, skip_fn=lambda n: n in skip)

with PRIORITY_PATH.open('w', encoding='utf-8') as f:
    json.dump(priority_order, f, indent=2)

# `priority_order` now contains a prioritized list of documentation targets

Skipped  9761, Scheduled    44 nodes with fan-in 0. Total visited: 9805/14517
Skipped     0, Scheduled     6 nodes with fan-in 0. Total visited: 9811/14517
Skipped     0, Scheduled     1 nodes with fan-in 0. Total visited: 9812/14517
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 9813/14517
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 9814/14517
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 9815/14517
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 9816/14517
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 9817/14517
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 9818/14517
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 9819/14517
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 9820/14517
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 9821/14517
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visite

In [7]:
# clear all responses in the database
if False:
    for cid, docs in doc_db.docs.items():
        for sig, doc in docs.items():
            if 'response' in doc and not isinstance(doc['response'], dict):
                del doc['response']
        doc_db.save_docs(cid)

In [8]:
# Refinement pass: Enhance documentation with usage examples using Jinja2
from jinja2 import Environment, FileSystemLoader
from llm_utils import call_openai
import json
import re

OPENAI_MODEL = "gpt-4.1-nano"
OPENAI_MAX_TOKENS = None

# Shared environment setup for Jinja2
template_dir = PROJECT_ROOT / ".ai/templates"
env = Environment(loader=FileSystemLoader(template_dir))

def render_template(template_name: str, **context) -> str:
    template = env.get_template(template_name)
    return template.render(**context)

def qualified_kind(doc: doc_db.Document) -> str:
    if doc.kind == 'define':
        return 'macro'
    if doc.kind == 'variable' and doc.cid.startswith('group_'):
        return 'constant'
    return doc.kind

def qualified_name(doc: doc_db.Document) -> str:
    if doc.kind in ('function', 'variable', 'enum'):
        if any(doc.cid.startswith(k) for k in ('class', 'struct', 'namespace', 'group')):
            compound = db.get(doc.cid)
            return f"{compound.name}::{doc.name}"
    return doc.name

def build_signature(doc: doc_db.Document) -> str:
    signature = doc.name or doc.definition
    signature = signature.replace(doc.name, qualified_name(doc))
    if doc.argsstring:
        signature += doc.argsstring
    return signature

def build_existing_doc_dict(doc: doc_db.Document) -> Dict[str, str|Dict[str, str]] | None:
    if any([doc.brief, doc.details, doc.params, doc.returns, doc.tparams, doc.throws, doc.notes, doc.rationale]):
        return {
            'brief': doc.brief,
            'details': doc.details,
            'params': doc.params,
            'returns': doc.returns,
            'tparams': doc.tparams,
            'throws': doc.throws,
            'notes': doc.notes,
            'rationale': doc.rationale
        }
    return None

def format_dep_summary(doc: doc_db.Document) -> str:

    # if kind in ('function', 'friend'):
    #     name = doc.definition + doc.argsstring
    # else:
    #     name = doc.name
    if doc.response:
        # print(f"using improved summary documentation for {doc.name}")
        brief = doc.response.brief
        details = doc.response.details
        params = doc.response.params
        tparams = doc.response.tparams
        throws = doc.response.throws
        returns = doc.response.returns
        notes = doc.response.notes
        rationale = doc.response.rationale
    else:
        brief = doc.brief
        details = doc.details
        params = doc.params
        tparams = doc.tparams
        throws = doc.throws
        returns = doc.returns
        notes = doc.notes
        rationale = doc.rationale

    brief = brief or details # could be pulling from code

    summary = f"[{qualified_kind(doc)}] {qualified_name(doc)}"
    if brief:
        summary += f" - {brief.strip()}".rstrip('. ')
    if rationale:
        summary += f".  Rationale: {rationale.strip()}".rstrip('. ')

    if doc.kind in ('function', 'friend'):
        if params:
            param_str = '; '.join([f'{param.strip()} - {desc.strip()}'.rstrip('. ') for param, desc in params.items()])
        else:
            param_str = '; '.join(p.strip() for p in doc.argsstring.strip('() ').split(','))

        returns = returns or doc.definition.rsplit(' ', 1)[0]
        if params:
            summary += f".  Params: {param_str}"
        if returns:
            summary += f".  Returns: {returns}"

    return summary

def format_usages(usages: Dict[str, str]) -> Dict[str, str]:
    if not usages:
        return None
    formatted_usages = {}
    for context, description in sorted(usages.items(), key=lambda x: len(x[1]), reverse=True)[:5]:
        try:
            _, function_signature = context.split(', ', 1)
            formatted_usages[function_signature] = description
        except ValueError:
            formatted_usages[context] = description
    return formatted_usages

def create_refinement_prompt(doc: doc_db.Document, code: str, dep_docs: list[doc_db.Document]) -> str:
    template_name = {
        'function': 'docgen_fw_refine_function.j2',
        'variable': 'docgen_fw_refine_variable.j2',
        'define': 'docgen_fw_refine_variable.j2',
        'enum': 'docgen_fw_refine_enum.j2',
        'class': 'docgen_fw_refine_class.j2',
        'struct': 'docgen_fw_refine_class.j2',
        'namespace': 'docgen_fw_refine_group.j2',
        'group': 'docgen_fw_refine_group.j2'
    }.get(doc.kind, 'docgen_fw_refine_function.j2')

    context = {
        'kind': qualified_kind(doc),
        'name': doc.name,
        'signature': build_signature(doc),
        'code': code,
        'dep_summaries': [format_dep_summary(doc) for doc in dep_docs],
        'usages': format_usages(doc.usages),
        'existing_doc': build_existing_doc_dict(doc)
    }
    return render_template(template_name, **context)


In [9]:
from json import JSONDecodeError
from pydantic import ValidationError
from llm_utils import call_openai
from gen_common import document_entity, get_dependency_entities

OPENAI_MODEL = "gpt-4.1-nano"
OPENAI_MAX_TOKENS = None

skip_completed = True
dry_run = False

# Count nodes in the graph whose "kind" is in the specified list

nodes_to_document: List[Tuple[str, dp.DoxygenEntity]] = []
for item in priority_order:
    node_id, kind, fan_in = item['id'], item['kind'], item['fan_in']

    eid = dg.get_body_eid(db, node_id)
    entity = db.get(eid)

    doc = doc_db.get_doc(entity.id.compound, entity.signature)

    if doc and skip_completed and doc.state == doc_db.DocumentState.REFINED_SUMMARY:
        continue

    nodes_to_document.append((node_id, entity, fan_in))

completed = 0

# --- Forward pass cell ---
# nodes_to_document = list(reversed(nodes_to_document))
for node_id, entity, fan_in in nodes_to_document:
    # if entity.kind != 'class':
    #     continue

    completed += 1
    print(f"{completed:4}/{len(nodes_to_document):4}: {str(entity.id)} - {entity.signature}")

    code_data = dg.get_code_body(PROJECT_ROOT, db, graph, node_id)
    if not code_data or not code_data['code']:
        print(f"* warning: failed to retrieve code body for node_id: {node_id}")
        continue

    body = code_data['code']
    doc = doc_db.get_doc(entity.id.compound, entity.signature)

    if not doc:
        doc = document_entity(entity)

    dep_ids = dg.fan_in(graph, node_id, [dg.Relationship.REQUIRED_BY, dg.Relationship.CONTAINED_BY])
    deps = [db.get(dg.get_body_eid(db, dep_id)) for dep_id in dep_ids]
    # deps2 = get_dependency_entities(db, graph, node_id)
    # print(len(deps), len(deps2))
    dep_docs = [doc_db.get_doc(dep.id.compound, dep.signature) for dep in deps]

    prompt = create_refinement_prompt(doc, body, dep_docs)
    # print(prompt)

    if dry_run:
        doc_json = "(dry run)"
    else:
        response_text = call_openai(prompt, OPENAI_MODEL, OPENAI_MAX_TOKENS)
        try:
            response_text = response_text.lstrip("```json").rstrip("```")
            # Add an extra backslash to single backslashes not part of an escaped quote or escaped single quote
            response_text = re.sub(r'\\(?!["\'])', r'\\\\', response_text)
            doc_json = json.loads(response_text)
            doc_json = {k: v for k, v in doc_json.items() if v}
            # pprint(doc_json)
            doc.response = doc_db.DoxygenFields(**doc_json)
            doc.prompt = prompt
            doc.state = doc_db.DocumentState.REFINED_SUMMARY
            doc_db.save_docs(entity.id.compound)
        except JSONDecodeError as e:
            print(response_text)
            print(f"Failed to parse model output as JSON: {e}")
        except ValidationError as e:
            pprint(doc_json)
            print(f"Failed to validate model output: {e}")
    
    # if fan_in > 0:
    #     break

print("[✓] Forward pass complete.")


   1/2972: namespacestd - std
* warning: failed to retrieve code body for node_id: namespacestd
   2/2972: class_actable_1a314989f698e986138ae182eb58d73411 - virtual const std::string Actable::identifier() const =0
* warning: failed to retrieve code body for node_id: 1a314989f698e986138ae182eb58d73411
   3/2972: classworldmap_1_1_quadtree_1ae6d6420e89a488e94a6537a9e900128e - worldmap::Quadtree< T >::Quadtree()
* warning: failed to retrieve code body for node_id: 1ae6d6420e89a488e94a6537a9e900128e
   4/2972: group___channel_flags_1gaabd3e6481152660cbe87acf88188b1d5 - CHAN_QWEST
   5/2972: group___color_escape_constants_1ga405c715bbd1fc444438442528bbe9e91 - C_B_RED
   5/2972: group___color_escape_constants_1ga405c715bbd1fc444438442528bbe9e91 - C_B_RED
   6/2972: structattack__type_1a941b9f29bc13611355baaed1cede45a9 - damage
   6/2972: structattack__type_1a941b9f29bc13611355baaed1cede45a9 - damage
   7/2972: group___color_slot_types_1ga37ae693283d3f8790f0c05c09d57435b - CSLOT_SCORE_XPNUM


In [11]:
# commit changes to docs
def commit_docs():
    for cid, cid_docs in doc_db.docs.items():
        for sig, doc in cid_docs.items():
            if not doc.response:
                continue

            print(f"updating doc {cid} {sig}")
            doc.brief = doc.response.brief or doc.brief
            doc.details = doc.response.details or doc.details
            if doc.response.params:
                doc.params = doc.params or {}
                doc.params.update(doc.response.params)
            if doc.response.tparams:
                doc.tparams = doc.tparams or {}
                doc.tparams.update(doc.response.tparams)
            doc.returns = doc.response.returns or doc.returns
            doc.throws = doc.response.throws or doc.throws
            doc.notes = doc.response.notes or doc.notes
            doc.rationale = doc.response.rationale or doc.rationale

            doc.response = None
            doc.prompt = None

        doc_db.save_docs(cid)

commit_docs()

updating doc structgem_1_1type__st gem::type_st
updating doc structgem_1_1type__st keyword
updating doc structgem_1_1type__st color_code
updating doc structgem_1_1type__st vnum
updating doc structgem_1_1type__st apply_loc
updating doc structgem_1_1type__st modifier
updating doc group___loot_armor_vnums LootArmorVnums
updating doc group___loot_armor_vnums OBJ_VNUM_LIGHT
updating doc group___loot_armor_vnums OBJ_VNUM_FINGER
updating doc group___loot_armor_vnums OBJ_VNUM_NECK
updating doc group___loot_armor_vnums OBJ_VNUM_TORSO
updating doc group___loot_armor_vnums OBJ_VNUM_HEAD
updating doc group___loot_armor_vnums OBJ_VNUM_LEGS
updating doc group___loot_armor_vnums OBJ_VNUM_FEET
updating doc group___loot_armor_vnums OBJ_VNUM_HANDS
updating doc group___loot_armor_vnums OBJ_VNUM_ARMS
updating doc group___loot_armor_vnums OBJ_VNUM_SHIELD
updating doc group___loot_armor_vnums OBJ_VNUM_BODY
updating doc group___loot_armor_vnums OBJ_VNUM_WAIST
updating doc group___loot_armor_vnums OBJ_VNUM_WR